# Exercise 3 - Narbonne Film Festival Format Validator

## Overview

This notebook automates the format validation and conversion pipeline for the
**Narbonne Online Film Festival**. Every year the festival receives over 100 films,
some of which are not submitted in the required digital format. This application
examines each submitted film using `ffprobe`, identifies non-compliant fields,
generates a written report, and automatically re-encodes non-compliant files
using `ffmpeg`.

## How the Pipeline Works?

**Tool Check** - verifies that `ffprobe` and `ffmpeg` are installed and accessible

**Load Specifications** - defines the 9 required format fields in `target_specs_required`

**Probe Files** - `ffprobe` reads each file's container, codec, resolution, frame
rate, bitrate, and audio properties as JSON

**Validate** - compares each property against `target_specs_required` and collects a list
of problematic fields

**Report** - writes `outputs/format_audit_report.txt` listing every non-compliant
file and its specific issues

**Convert** - `ffmpeg` re-encodes each non-compliant file to the exact festival
specification, saving it as `<name>_formatOK.mp4`

**Verify** - re-probes every converted file to confirm full compliance

## Key Parameters

`container = mp4` - required file wrapper format

`video_codec = hevc` - H.265 compression (High Efficiency Video Coding)

`audio_codec = aac` - Advanced Audio Coding

`fps = 25` - frames displayed per second

`aspect = 16:9` - width-to-height display ratio

`resolution = 640 x 360` - pixel dimensions of each frame

`video_bitrate = 2 – 5 Mb/s` - data rate for video stream

`audio_bitrate = up to 256 kb/s` - data rate for audio stream

`audio_channels = 2` - stereo (left and right speaker)

## Input & Output

- Input files: `Exercise3_Files/` - original submitted films (mp4, mov, avi, mkv)
- Output report: `outputs/format_audit_report.txt`
- Output videos: `outputs/converted/*_formatOK.mp4`

## Import Libraries

In [1]:
import json
import subprocess
from pathlib import Path
from math import gcd
import glob
import subprocess
import platform

## FFMPEG Verification & Installation

In [2]:
def check_and_install_ffmpeg():

    # Check if ffmpeg is already installed
    try:
        result = subprocess.run(
            ["ffmpeg", "-version"],
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print("ffmpeg already installed")
            return
    except FileNotFoundError:
        print("ffmpeg not found. Attempting to install!\n")

    os_type = platform.system()

    if os_type == "Darwin":
        # macOS
        try:
            print("macOS detected. Installing via Homebrew!")
            subprocess.run(["brew", "install", "ffmpeg"], check=True)
            print("ffmpeg installed via Homebrew")
        except Exception as e:
            print(f"Homebrew installation failed: {e}")

    elif os_type == "Linux":
        # Linux
        try:
            print("Linux detected. Installing via apt-get!")
            subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
            print("ffmpeg installed via apt-get!")
        except Exception as e:
            print(f"apt-get installation failed: {e}")

    else:
        # Windows
        print("Windows detected. Manual installation required!")
        print("Download from official site:")
        print("https://ffmpeg.org/download.html")
        print("Use This Link As Guide If Not Sure With Installation: https://www.youtube.com/watch?v=KBnyOH1o5Ms")

check_and_install_ffmpeg()

# Verify both tools are accessible before running pipeline
try:
    result = subprocess.run(["ffprobe", "-version"], capture_output=True, text=True)
    ffprobe_bin = "ffprobe" if result.returncode == 0 else None
except FileNotFoundError:
    ffprobe_bin = None

try:
    result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
    ffmpeg_bin = "ffmpeg" if result.returncode == 0 else None
except FileNotFoundError:
    ffmpeg_bin = None

if not ffprobe_bin or not ffmpeg_bin:
    raise EnvironmentError("ffmpeg/ffprobe not available. Install before continuing!")

print(f"ffprobe: {ffprobe_bin} | ffmpeg: {ffmpeg_bin}")

ffmpeg already installed
ffprobe: ffprobe | ffmpeg: ffmpeg


## Festival Specifications & Functions
This cell defines the target format specifications required by the festival, followed by all the helper functions needed for the pipeline.

In [3]:
# Festival Specifications
target_specs_required = {
    "container":         "mp4",
    # h.265
    "video_codec":       "hevc",
    "audio_codec":       "aac",
    "fps":               25.0,
    "aspect":            "16:9",
    "resolution":        (640, 360),
    # 2 Mb/s
    "video_bitrate_min": 2_000_000,
    # 5 Mb/s
    "video_bitrate_max": 5_000_000,
    # 256 kb/s
    "audio_bitrate_max": 256_000,
    "audio_channels":    2,
}

print("FESTIVAL SPECIFICATIONS:")
for key, value in target_specs_required.items():
    print(f"  {key:<22}: {value}")

# Helper Functions
def parse_fps(ratio):
    if not ratio or ratio == "0/0":
        return 0.0
    num, den = ratio.split("/")
    return float(num) / float(den)

def calc_aspect_ratio(width, height):
    common = gcd(width, height)
    return f"{width // common}:{height // common}"

def bitrate_to_mbps(bitrate):
    return f"{bitrate / 1_000_000:.2f} Mb/s"

def bitrate_to_kbps(bitrate):
    return f"{bitrate / 1_000:.1f} kb/s"

# ffprobe wrapper
def probe_file(filepath):
    cmd = [
        ffprobe_bin,
        "-v",
        "error",
        "-show_entries",
        "format=format_name,bit_rate:"
        "stream=index,codec_type,codec_name,width,height,"
        "r_frame_rate,bit_rate,channels",
        "-of",
        "json",
        str(filepath),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed: {result.stderr}")
    return json.loads(result.stdout)

# Validation
def validate_file(filepath):
    issues   = []
    data     = probe_file(filepath)
    v_stream = next((s for s in data["streams"] if s.get("codec_type") == "video"), None)
    a_stream = next((s for s in data["streams"] if s.get("codec_type") == "audio"), None)

    # 1.Container
    format_name = str(data.get("format", {}).get("format_name", ""))
    if target_specs_required["container"] not in format_name:
        issues.append(f"Container: expected mp4, got '{format_name}'")

    if v_stream:
        # 2.Video codec
        v_codec = v_stream.get("codec_name", "")
        if v_codec != target_specs_required["video_codec"]:
            issues.append(f"Video codec: expected hevc, got '{v_codec}'")

        # 3.Resolution
        w = int(v_stream.get("width")  or 0)
        h = int(v_stream.get("height") or 0)
        if (w, h) != target_specs_required["resolution"]:
            issues.append(f"Resolution: expected 640x360, got {w}x{h}")

        # 4.Fps
        fps = parse_fps(str(v_stream.get("r_frame_rate", "0/0")))
        if abs(fps - target_specs_required["fps"]) > 0.15:
            issues.append(f"FPS: expected ~25, got {fps:.2f}")

        # 5.Aspect ratio
        if h > 0:
            aspect = calc_aspect_ratio(w, h)
            if aspect != target_specs_required["aspect"]:
                issues.append(f"Aspect ratio: expected 16:9, got {aspect}")

        # 6.Video bitrate
        vbr = int(v_stream.get("bit_rate") or 0)
        if not (target_specs_required["video_bitrate_min"] <= vbr <= target_specs_required["video_bitrate_max"]):
            issues.append(f"Video bitrate: expected 2-5 Mb/s, got {bitrate_to_mbps(vbr)}")

    if a_stream:
        # 7.Audio codec
        a_codec = a_stream.get("codec_name", "")
        if a_codec != target_specs_required["audio_codec"]:
            issues.append(f"Audio codec: expected aac, got '{a_codec}'")

        # 8.Audio channels
        channels = int(a_stream.get("channels") or 0)
        if channels != target_specs_required["audio_channels"]:
            issues.append(f"Audio channels: expected 2, got {channels}")

        # 9.Audio bitrate
        abr = int(a_stream.get("bit_rate") or 0)
        if abr > target_specs_required["audio_bitrate_max"]:
            issues.append(f"Audio bitrate: expected ≤256 kb/s, got {bitrate_to_kbps(abr)}")

    return len(issues) == 0, issues

# Conversion
def convert_file(input_path, output_path):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        ffmpeg_bin, "-y", "-i", str(input_path),
        "-vf",
        "scale=640:360,fps=25,setsar=1",
        "-c:v",
        "libx265",
        "-b:v",
        "3M",
        "-maxrate",
        "5M",
        "-bufsize",
        "6M",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        "-b:a",
        "192k",
        "-ac",
        "2",
        str(output_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0

FESTIVAL SPECIFICATIONS:
  container             : mp4
  video_codec           : hevc
  audio_codec           : aac
  fps                   : 25.0
  aspect                : 16:9
  resolution            : (640, 360)
  video_bitrate_min     : 2000000
  video_bitrate_max     : 5000000
  audio_bitrate_max     : 256000
  audio_channels        : 2


## Process & Validate Films
This cell scans the `Exercise3_Files/` folder, validates each video against the festival specifications, saves the results to `format_audit_report.txt` and automatically converts every non-compliant file.

In [4]:
# Locate all input video files
video_files = (
    glob.glob("Exercise3_Files/*.mp4") +
    glob.glob("Exercise3_Files/*.mov") +
    glob.glob("Exercise3_Files/*.avi") +
    glob.glob("Exercise3_Files/*.mkv")
)

if not video_files:
    print("No files found")
else:
    print(f"PROCESSING {len(video_files)} FILES:\n")

    Path("outputs/converted").mkdir(parents=True, exist_ok=True)

    invalid_files = []
    report_lines  = []
    report_lines.append("NARBONNE FILM FESTIVAL — FORMAT VALIDATION REPORT\n")
    report_lines.append("=" * 60 + "\n\n")

    # Validate each file
    for filepath in sorted(video_files):
        filename = Path(filepath).name
        print(f"Checking: {filename}")

        is_valid, issues = validate_file(filepath)

        if is_valid:
            print("VALID\n")
            report_lines.append(f"File: {filename}\n")
            report_lines.append("Compliant : YES\n")
            report_lines.append("Problematic fields: None\n\n")
        else:
            print(f"INVALID ({len(issues)} issues)")
            for issue in issues:
                print(f"~ {issue}")
            print()
            invalid_files.append(filepath)
            report_lines.append(f"File: {filename}\n")
            report_lines.append("Compliant: NO\n")
            report_lines.append("Problematic fields:\n")
            for issue in issues:
                report_lines.append(f"- {issue}\n")
            converted_name = f"{Path(filepath).stem}_formatOK.mp4"
            report_lines.append(f"Converted: {converted_name}\n\n")

    # Save txt report
    report_path = Path("outputs") / "format_audit_report.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.writelines(report_lines)
    print(f"Report saved: {report_path}")

    # Convert all invalid files
    print(f"\nCONVERTING {len(invalid_files)} FILES:\n")
    success_count = 0
    for filepath in invalid_files:
        output_path = f"outputs/converted/{Path(filepath).stem}_formatOK.mp4"
        print(f"Converting: {Path(filepath).name}")
        success = convert_file(filepath, output_path)
        if success:
            print(f"Saved: {Path(output_path).name}\n")
            success_count += 1
        else:
            print(f"Failed!\n")

    print(f"{success_count}/{len(invalid_files)} converted successfully!")

PROCESSING 5 FILES:

Checking: Cosmos_War_of_the_Planets.mp4
INVALID (5 issues)
~ Video codec: expected hevc, got 'h264'
~ Resolution: expected 640x360, got 628x354
~ FPS: expected ~25, got 29.97
~ Aspect ratio: expected 16:9, got 314:177
~ Audio bitrate: expected ≤256 kb/s, got 317.1 kb/s

Checking: Last_man_on_earth_1964.mov
INVALID (5 issues)
~ Video codec: expected hevc, got 'prores'
~ FPS: expected ~25, got 23.98
~ Video bitrate: expected 2-5 Mb/s, got 9.29 Mb/s
~ Audio codec: expected aac, got 'pcm_s16le'
~ Audio bitrate: expected ≤256 kb/s, got 1536.0 kb/s

Checking: The_Gun_and_the_Pulpit.avi
INVALID (7 issues)
~ Container: expected mp4, got 'avi'
~ Video codec: expected hevc, got 'rawvideo'
~ Resolution: expected 640x360, got 720x404
~ Aspect ratio: expected 16:9, got 180:101
~ Video bitrate: expected 2-5 Mb/s, got 87.44 Mb/s
~ Audio codec: expected aac, got 'pcm_s16le'
~ Audio bitrate: expected ≤256 kb/s, got 1536.0 kb/s

Checking: The_Hill_Gang_Rides_Again.mp4
INVALID (2 iss

## Verifying Converted Files
To confirm the conversion worked correctly, each `<video_name>_formatOK.mp4` file is re-probed using `ffprobe` and re-validated against `TARGET_SPEC`.

In [5]:
print("VERIFYING CONVERTED FILES\n")

format_ok_files = sorted(glob.glob("outputs/converted/*_formatOK.mp4"))
perfect_count   = 0

for filepath in format_ok_files:
    print(f"{Path(filepath).name}")

    is_valid, issues = validate_file(filepath)

    data     = probe_file(filepath)
    v_stream = next((s for s in data["streams"] if s["codec_type"] == "video"), None)
    a_stream = next((s for s in data["streams"] if s["codec_type"] == "audio"), None)

    if v_stream:
        vbr = int(v_stream.get("bit_rate") or 0)
        print(f"Container    : {data['format'].get('format_name')}")
        print(f"Video codec  : {v_stream['codec_name']}")
        print(f"Resolution   : {v_stream['width']}x{v_stream['height']}")
        print(f"FPS          : {parse_fps(v_stream['r_frame_rate']):.2f}")
        print(f"Video bitrate: {bitrate_to_mbps(vbr)}")
    if a_stream:
        abr = int(a_stream.get("bit_rate") or 0)
        print(f"Audio codec  : {a_stream['codec_name']}")
        print(f"Channels     : {a_stream.get('channels')}")
        print(f"Audio bitrate: {bitrate_to_kbps(abr)}")

    if is_valid:
        print("100% FESTIVAL COMPLIANT!")
        perfect_count += 1
    else:
        print("Issues remaining:")
        for issue in issues:
            print(f"~ {issue}")
    print("-" * 55)

# Final summary
report_exists = Path("outputs/format_audit_report.txt").exists()
print(f"\nRESULT : {perfect_count}/{len(format_ok_files)} PERFECT")
print(f"Report : {'EXISTS' if report_exists else 'MISSING'}")
print(f"Files  : {len(format_ok_files)} converted")

VERIFYING CONVERTED FILES

Cosmos_War_of_the_Planets_formatOK.mp4
Container    : mov,mp4,m4a,3gp,3g2,mj2
Video codec  : hevc
Resolution   : 640x360
FPS          : 25.00
Video bitrate: 3.05 Mb/s
Audio codec  : aac
Channels     : 2
Audio bitrate: 193.4 kb/s
100% FESTIVAL COMPLIANT!
-------------------------------------------------------
Last_man_on_earth_1964_formatOK.mp4
Container    : mov,mp4,m4a,3gp,3g2,mj2
Video codec  : hevc
Resolution   : 640x360
FPS          : 25.00
Video bitrate: 3.29 Mb/s
Audio codec  : aac
Channels     : 2
Audio bitrate: 192.4 kb/s
100% FESTIVAL COMPLIANT!
-------------------------------------------------------
The_Gun_and_the_Pulpit_formatOK.mp4
Container    : mov,mp4,m4a,3gp,3g2,mj2
Video codec  : hevc
Resolution   : 640x360
FPS          : 25.00
Video bitrate: 2.91 Mb/s
Audio codec  : aac
Channels     : 2
Audio bitrate: 192.5 kb/s
100% FESTIVAL COMPLIANT!
-------------------------------------------------------
The_Hill_Gang_Rides_Again_formatOK.mp4
Container 